# Tầng 1 – So sánh Naive Bayes / Linear SVM / CharCNN (ham · spam · scam)

Notebook này train **3 mô hình** trên **3 thiết lập** (Tiếng Việt, Tiếng Anh, Song ngữ) với **đúng dữ liệu đã chốt**
(nhãn v3, chia theo họ mẫu, external VI crawl lần 2), rồi **chọn mô hình tầng 1** theo quy tắc cố định:

> **F2 lớp scam cao nhất trên validation** (hòa → macro-F1). Test và external **chỉ để báo cáo**, không dùng để chọn.

Tầng 2 (scam → loại scam) làm sau, chỉ chạy khi tầng 1 đoán `scam`.

**Cách chạy:** bấm *Run all* (Colab: `Runtime → Run all`; Kaggle: `Run All`). Thời gian khoảng 15–30 phút với GPU.

**Chuẩn bị:**
- **Colab:** `Runtime → Change runtime type → T4 GPU`. Ô bước 2 sẽ hỏi upload file `bilingual-scam-colab.zip`.
- **Kaggle:** `Settings → Accelerator → GPU T4`; `Add Input → Upload` file `bilingual-scam-colab.zip` (dataset).

Kết thúc notebook tạo file **`tier1_results.zip`** – tải về và gửi lại để cập nhật kết quả vào dự án.

## Bước 1 – Kiểm tra GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: đang chạy CPU – CharCNN sẽ rất chậm. Hãy bật GPU rồi chạy lại từ đầu.")

## Bước 2 – Lấy mã nguồn + dữ liệu (tự nhận Colab / Kaggle)

In [ ]:
import glob, os, shutil, zipfile

ZIP_NAME = "bilingual-scam-colab.zip"
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
PROJECT = f"{WORK}/bilingual-scam-baseline"

if not os.path.exists(PROJECT):
    if ON_KAGGLE:
        zips = glob.glob(f"/kaggle/input/**/{ZIP_NAME}", recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall(WORK)
        else:  # Kaggle tự giải nén zip khi tạo dataset
            found = glob.glob("/kaggle/input/**/bilingual-scam-baseline", recursive=True)
            assert found, "Không thấy dữ liệu: hãy Add Input file bilingual-scam-colab.zip"
            shutil.copytree(found[0], PROJECT)
    else:
        from google.colab import files
        print("Chọn file", ZIP_NAME)
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall(WORK)

os.chdir(PROJECT)
print("Thư mục làm việc:", os.getcwd())
for d in ["data/processed_vi_balanced", "data/processed_en_balanced", "data/processed_bilingual"]:
    print(d, "OK" if os.path.exists(f"{d}/modeling_data.csv") else "THIẾU")

## Bước 3 – Cài thư viện

In [ ]:
!pip install -q -e .
# Kernel đang chạy không thấy gói vừa cài kiểu editable (chỉ có hiệu lực khi khởi động lại),
# nên thêm src/ vào đường dẫn cho cả notebook lẫn các lệnh !python bên dưới.
import sys
SRC = os.path.join(PROJECT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
import scam_baseline, sklearn
print("scikit-learn", sklearn.__version__, "| scam_baseline OK")

## Bước 4 – Kiểm tra dữ liệu đúng bản đã chốt
Số dòng train phải khớp với máy local (nếu khác → zip cũ, hãy đóng gói lại).

In [ ]:
import pandas as pd
EXPECTED_TRAIN = {"vietnamese_balanced": 4899, "bilingual": 13018}
SETUPS = {"vietnamese_balanced": "data/processed_vi_balanced",
          "english_balanced": "data/processed_en_balanced",
          "bilingual": "data/processed_bilingual"}
for setup, path in SETUPS.items():
    df = pd.read_csv(f"{path}/modeling_data.csv", usecols=["split", "label_level_1"], low_memory=False)
    counts = df.split.value_counts().to_dict()
    flag = ""
    if setup in EXPECTED_TRAIN:
        flag = "khớp" if counts.get("train") == EXPECTED_TRAIN[setup] else f"KHÁC (mong đợi {EXPECTED_TRAIN[setup]})"
    print(f"{setup:20s} train={counts.get('train')} val={counts.get('validation')} test={counts.get('test')} "
          f"external={counts.get('external_test')}  {flag}")

## Bước 5 – Train 3 mô hình × 3 thiết lập
NB và SVM train lại trên cùng dữ liệu (kết quả giống bản local) để cả 3 mô hình được đánh giá trong cùng một lần chạy.
CharCNN: tối đa 30 epoch, **5 epoch khởi động** không được chọn / không tính dừng sớm, sau đó dừng nếu 5 epoch liền không cải thiện **F2 lớp scam trên validation** (bản 26/11 lần 2 – lần 1 dừng ở epoch 4 vì chọn nhầm epoch 1).

In [ ]:
import time
for setup in ["vietnamese_balanced", "english_balanced", "bilingual"]:
    start = time.time()
    print(f"\n===== {setup} =====")
    !python -m scam_baseline.cli --config configs/{setup}.yaml train --models nb svm charcnn
    print(f"--> {setup}: {(time.time() - start) / 60:.1f} phút")

## Bước 6 – Đường học của CharCNN (F2 scam trên validation theo epoch)

In [ ]:
import json
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, setup in zip(axes, ["vietnamese_balanced", "english_balanced", "bilingual"]):
    hist = json.load(open(f"artifacts/{setup}/charcnn/training_history.json"))
    meta = json.load(open(f"artifacts/{setup}/charcnn/training_metadata.json"))
    ep = [h["epoch"] for h in hist]
    ax.plot(ep, [h.get("validation_scam_f2") for h in hist], marker="o", label="val F2 scam")
    ax.plot(ep, [h["validation_macro_f1"] for h in hist], marker=".", label="val macro-F1")
    ax.axvspan(0.5, meta.get("min_epochs", 1) - 0.5, color="#eee", label="khởi động")
    ax.axvline(meta["best_epoch"], color="gray", linestyle="--", label=f"chọn epoch {meta['best_epoch']}")
    ax.set_title(setup); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Bước 7 – Chấm tập scam tiếng Việt thật đầy đủ
Tập đầy đủ gồm cả tin cùng khuôn với tập Tuấn và tin Zalo/Messenger (không nằm trong pipeline – xem nhật ký mục 2.8).

In [ ]:
!python scripts/score_vi_crawl_full.py > /dev/null
full = pd.read_csv("reports/vi_crawl_full/summary.csv")
view = full[full.subset.isin(["sms_new_family", "sms_known_family", "chat"])]
display(view.pivot(index="model", columns="subset", values="scam_recall").round(3))

## Bước 8 – So sánh và chọn mô hình tầng 1

In [ ]:
!python scripts/compare_tier1.py > /dev/null
from IPython.display import Markdown, display
display(Markdown(open("reports/tier1/tier1_selection.md", encoding="utf-8").read()))

### Song ngữ: kết quả tách theo ngôn ngữ (test)
Nếu dùng **1 model song ngữ** thay cho 2 model riêng, phần tiếng Việt/tiếng Anh của nó phải không kém model riêng.

In [ ]:
cmp = pd.read_csv("reports/tier1/tier1_comparison.csv")
cols = ["Thiết lập", "Mô hình", "selected", "val_scam_f2", "test_scam_f2", "test_vi_scam_f2", "test_en_scam_f2",
        "vi_crawl_scam_recall", "vie_sms_scam_fpr", "smishx_scam_recall", "smishx_scam_fpr"]
display(cmp[[c for c in cols if c in cmp.columns]].round(3))

## Bước 9 – Đóng gói kết quả để gửi về

In [ ]:
import zipfile
OUT = f"{WORK}/tier1_results.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for setup in ["vietnamese_balanced", "english_balanced", "bilingual"]:
        for path in glob.glob(f"artifacts/{setup}/**/*", recursive=True) + glob.glob(f"reports/{setup}/**/*", recursive=True):
            if os.path.isfile(path):
                z.write(path)
    for path in glob.glob("reports/tier1/*") + glob.glob("reports/vi_crawl_full/*"):
        z.write(path)
print(OUT, f"{os.path.getsize(OUT) / 1e6:.1f} MB")
if not ON_KAGGLE:
    from google.colab import files
    files.download(OUT)
else:
    print("Kaggle: tải file ở tab Output (/kaggle/working/tier1_results.zip)")